# Phase 37: Chạy lại CAST dưới protocol của bài S2I (review 6, C2)

Cách dùng:
1. **Runtime → Restart session** (để chắc không còn tiến trình cũ).
2. Chạy lần lượt: cell 1 → các cell `%%writefile` → **37A**.
3. Khi muốn xem kết quả (kể cả lúc 37A chưa xong): bấm ■ dừng 37A, chạy **37B**, rồi chạy lại 37A để làm tiếp. Không mất gì, vì tiến trình được lưu sau mỗi bệnh nhân.

37A làm hai việc, **mỗi lúc một tiến trình** (GPU T4 đã chạy 100 % với một tiến trình, nên chạy song song không nhanh hơn):
1. **Đối chứng dương:** chính code này với protocol gốc của CAST, trên sub-01 và sub-13, để kiểm tra bản port không lỗi. CAST gốc báo cáo 0.268 và 0.784. Khoảng 5 giờ.
2. **Các fold còn lại** theo thứ tự ưu tiên (đã ghi trong kế hoạch phân tích trước khi xem thêm kết quả): 07, 08 (đang dở), rồi 09, 13, 15 (bệnh nhân có common-mode), rồi 10, 11, 12, 14. Mỗi fold khoảng 3 giờ.

Có thể dừng bất cứ lúc nào. Bài sẽ báo cáo đúng những fold đã chạy xong.

In [ ]:
#@title 1. Mount Drive + đường dẫn
from google.colab import drive
drive.mount('/content/drive')
import os, sys
ROOT     = '/content/drive/MyDrive/iEEG_S2I'   #@param {type:"string"}
CODE_DIR = f'{ROOT}/code'
RUNS_DIR = f'{ROOT}/runs'
DATA_DIR = f'{ROOT}/data/s2i_v1'          # dữ liệu huấn luyện encoder của bài
NOQC     = f'{ROOT}/data/s2i_v1_noqc'     # ds004752 mức primary
DATA17P  = f'{ROOT}/data/scales_s2i/primary'
DATA33   = f'{ROOT}/data/parm33/n32'      # cohort 3, 32 kênh (Phase 33)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR
os.chdir(CODE_DIR)
!nvidia-smi --query-gpu=name --format=csv,noheader
DATA36 = f'{ROOT}/data/s2i_v1_noqc_strict'   # tạo ở Phase 36A

In [ ]:
%%writefile {CODE_DIR}/s2i/metrics.py
"""
Evaluation metrics, computed per contact on the concatenation of all test trials.

Differences vs. old compute_metrics():
* R^2 is NOT clipped at 0 (clipping max(R^2, 0) hid failures and inflated means).
* Band-limited waveform r and band envelope r are reported, so we can see in
  which frequency range information actually exists.
* Filtering is done per trial (no filter ringing across trial boundaries).
"""
import numpy as np
from scipy.signal import butter, hilbert, sosfiltfilt

BANDS = {"delta": (1, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 70)}


def _r_rows(P, T):
    P = P - P.mean(1, keepdims=True)
    T = T - T.mean(1, keepdims=True)
    den = np.sqrt((P * P).sum(1) * (T * T).sum(1)) + 1e-12
    return (P * T).sum(1) / den


def _r2_rows(P, T):
    ss_res = ((T - P) ** 2).sum(1)
    ss_tot = ((T - T.mean(1, keepdims=True)) ** 2).sum(1) + 1e-12
    return 1.0 - ss_res / ss_tot


def per_contact_metrics(preds, tgts, fs=200, bands=True):
    """preds/tgts: lists of (K, L_trial) arrays. Returns dict of (K,) arrays."""
    P = np.concatenate(preds, 1).astype(np.float64)
    T = np.concatenate(tgts, 1).astype(np.float64)
    out = {"r": _r_rows(P, T), "r2": _r2_rows(P, T),
           "rmse": np.sqrt(((P - T) ** 2).mean(1))}
    if bands:
        for name, (lo, hi) in BANDS.items():
            sos = butter(4, [lo, min(hi, fs / 2 - 1)], btype="band", fs=fs, output="sos")
            pb, tb, pe, te = [], [], [], []
            for p, t in zip(preds, tgts):
                if p.shape[1] < 3 * 2 * len(sos) * 3:
                    continue
                fp, ft = sosfiltfilt(sos, p, axis=1), sosfiltfilt(sos, t, axis=1)
                pb.append(fp)
                tb.append(ft)
                pe.append(np.abs(hilbert(fp, axis=1)))
                te.append(np.abs(hilbert(ft, axis=1)))
            if pb:
                out[f"r_{name}"] = _r_rows(np.concatenate(pb, 1), np.concatenate(tb, 1))
                out[f"env_{name}"] = _r_rows(np.concatenate(pe, 1), np.concatenate(te, 1))
    return out


def summarize(m):
    return {k: float(np.nanmean(v)) for k, v in m.items()}


In [ ]:
%%writefile {CODE_DIR}/s2i/parm27.py
"""
Phase 27 - third cohort (Parmigiani et al. 2022; OSF wsgzp, CC BY-NC-SA 4.0), exactly as in
paper/cohort3_protocol.md (v1.0, 21 Oct 2026, written after the Phase 26 inventory and before any model was applied).

Commands
  build   extracted ccepcoreg tree (Phase 26) -> one s2i data file per included patient (format read by
          s2i.readout --strict) + p27_build.md / p27_build.json (execution log)
  runs    copy the final short encoder trained on ds004752 (s2i.shared --final --half 40 -> <enc_root>/short/
          enc_final.pt, the encoder used for ScaleOLDNEW) to <runs>/short/enc_<sub>.pt
  report  H1-H3 (two-sided Wilcoxon, Holm over H1-H3, median direction, patient bootstrap CI) + descriptive analyses
          -> p27_report.md / p27_report.json

Rules (protocol sections in brackets)
  runs [3]        a run is used only if its EEG-SEEG pairing is consistent: len(eeg_badtrials flags) = number of SEEG
                  epochs and the number of zero flags = number of EEG epochs (the k-th zero flag pairs SEEG epoch with
                  EEG epoch k); runs in run-number order (the only order available)
  channels [4]    EEG: status "good" in every used run; CAR over these channels. SEEG: bipolar channels as shared,
                  status "good" in every used run, sharing no contact with the stimulated pair of any used run
  signals [4]     each epoch (-0.3 ... +0.7 s at 1000 Hz) resampled to 200 Hz (scipy resample_poly 1/5), microvolts
  inclusion [3]   >= 100 paired epochs, >= 3 EEG channels, >= 3 valid targets (non-zero variance in calibration and test)
  split [5]       epochs in run order; calibration = first round(0.3 N); one-epoch gap; the rest is the test set
Storage as in s2i.scales17: robust-normalised with the patient's median / MAD of all kept samples, float16, with the
statistics (one copy per run = session); s2i.readout --strict re-normalises with the calibration epochs only.

python -m s2i.parm27 build  --src /content/parm --out DATA27
python -m s2i.parm27 runs   --enc_root ENC17 --data DATA27 --runs RUNS27
python -m s2i.parm27 report --data DATA27 --runs RUNS27
"""
import argparse
import glob
import json
import os
import re
import shutil
import sys
import time

import numpy as np

FS_IN, DOWN, FS = 1000, 5, 200
CAL_FRAC, MIN_EPOCHS, MIN_EEG, MIN_TGT = 0.3, 100, 3, 3


def log(msg):
    print(f"{time.strftime('%H:%M:%S')} | {msg}", flush=True)


def n_cal_of(n, frac=CAL_FRAC):
    return max(3, int(round(n * frac)))


def robust(x):
    med = np.median(x, axis=1, keepdims=True)
    mad = np.median(np.abs(x - med), axis=1, keepdims=True) * 1.4826
    return med, np.where(mad < 1e-30, 1.0, mad)


def read_tsv(f):
    lines = open(f, encoding="utf-8").read().splitlines()
    head = lines[0].split("\t")
    return [dict(zip(head, l.split("\t"))) for l in lines[1:] if l.strip()]


def contacts_of(ch):
    m = re.match(r"(\S+?)(\d+)-(\d+)$", ch)
    return (m.group(1), {int(m.group(2)), int(m.group(3))}) if m else (None, set())


def stim_pair(desc):
    m = re.search(r"Stimulation of\s+(\S+?)(\d+)-(\d+)\s", desc)
    return (m.group(1), {int(m.group(2)), int(m.group(3))}) if m else (None, set())


# ----------------------------------------------------------------------------- one patient
def patient_runs(root, bad, sid):
    """-> [(run, base_ieeg, base_eeg, flags, description)] with consistent pairing, and the list of skipped runs."""
    d = os.path.join(root, "derivatives", "epochs", sid)
    use, skip = [], []
    for fi in sorted(glob.glob(os.path.join(d, "ieeg", f"{sid}_task-ccepcoreg_run-*_epochs.npy")),
                     key=lambda p: int(re.search(r"run-(\d+)", p).group(1))):
        run = re.search(r"(run-\d+)", fi).group(1)
        fe = fi.replace(os.sep + "ieeg" + os.sep, os.sep + "eeg" + os.sep)
        flags = bad.get(sid, {}).get(run)
        if not os.path.exists(fe) or flags is None:
            skip.append((run, "missing EEG file or flags"))
            continue
        ni, ne = np.load(fi, mmap_mode="r").shape[0], np.load(fe, mmap_mode="r").shape[0]
        if len(flags) != ni or len(flags) - int(sum(flags)) != ne:
            skip.append((run, f"pairing inconsistent (SEEG {ni}, EEG {ne}, flags {len(flags)}/{len(flags) - int(sum(flags))})"))
            continue
        desc = json.load(open(fi[:-len("_epochs.npy")] + "_epochs.json")).get("Description", "")
        use.append((run, fi[:-len("_epochs.npy")], fe[:-len("_epochs.npy")], np.asarray(flags), desc))
    return use, skip


def build_patient(root, bad, sid):
    from scipy.signal import resample_poly
    info = {"subject": sid, "excluded": None}
    use, skip = patient_runs(root, bad, sid)
    info["runs_used"], info["runs_skipped"] = [u[0] for u in use], skip
    if not use:
        info["excluded"] = "no run with consistent pairing"
        return info, None
    chi = [read_tsv(u[1] + "_channels.tsv") for u in use]
    che = [read_tsv(u[2] + "_channels.tsv") for u in use]
    names_i, names_e = [c["name"] for c in chi[0]], [c["name"] for c in che[0]]
    assert all([c["name"] for c in x] == names_i for x in chi) and all([c["name"] for c in x] == names_e for x in che), \
        f"{sid}: channel lists differ between runs"
    good_e = [n for j, n in enumerate(names_e) if all(x[j].get("status") == "good" for x in che)]
    stims = [stim_pair(u[4]) for u in use]
    near = set()
    for n in names_i:
        sh, cs = contacts_of(n)
        if any(sh == s and cs & c for s, c in stims):
            near.add(n)
    good_i = [n for j, n in enumerate(names_i) if all(x[j].get("status") == "good" for x in chi) and n not in near]
    info.update(n_eeg_listed=len(names_e), n_eeg=len(good_e), n_seeg_listed=len(names_i),
                n_seeg_near_stim=len(near), n_seeg=len(good_i),
                stim_pairs=[f"{s}{'-'.join(map(str, sorted(c)))}" for s, c in stims])
    ie, ii = [names_e.index(n) for n in good_e], [names_i.index(n) for n in good_i]
    E, Y, ses, desc = [], [], [], []
    for run, bi, be, flags, d in use:
        xi = np.load(bi + "_epochs.npy", mmap_mode="r")
        xe = np.load(be + "_epochs.npy", mmap_mode="r")
        keep = np.where(flags == 0)[0]
        for k, j in enumerate(keep):                          # k-th EEG epoch <-> j-th SEEG epoch
            e = resample_poly(np.asarray(xe[k][ie], np.float64) * 1e6, 1, DOWN, axis=-1)
            y = resample_poly(np.asarray(xi[j][ii], np.float64) * 1e6, 1, DOWN, axis=-1)
            E.append(e - e.mean(0, keepdims=True))            # CAR over the kept EEG channels
            Y.append(y)
            ses.append(run)
        desc.append({"run": run, "description": d, "n_pairs": int(len(keep))})
    N = len(E)
    nc = n_cal_of(N)
    info.update(n_epochs=N, n_cal=nc, n_test=max(N - nc - 1, 0), calib_minutes=nc * E[0].shape[1] / FS / 60,
                test_minutes=max(N - nc - 1, 0) * E[0].shape[1] / FS / 60, samples_per_epoch=int(E[0].shape[1]),
                runs=desc, calib_runs=sorted(set(ses[:nc]), key=ses.index), test_runs=sorted(set(ses[nc + 1:]), key=ses.index))
    valid = [k for k in range(len(good_i))
             if np.var(np.concatenate([y[k] for y in Y[:nc]])) > 0 and np.var(np.concatenate([y[k] for y in Y[nc + 1:]])) > 0]
    info["n_valid_targets"] = len(valid)
    if N < MIN_EPOCHS:
        info["excluded"] = f"{N} paired epochs (< {MIN_EPOCHS})"
    elif len(good_e) < MIN_EEG:
        info["excluded"] = f"{len(good_e)} EEG channels (< {MIN_EEG})"
    elif len(valid) < MIN_TGT:
        info["excluded"] = f"{len(valid)} valid targets (< {MIN_TGT})"
    return info, (E, [y[valid] for y in Y], ses, good_e, [good_i[k] for k in valid])


def write_patient(a, root, sid, data):
    import torch
    E, Y, ses, eeg_names, bip_names = data
    mni = {}
    f = glob.glob(os.path.join(root, "derivatives", "epochs", sid, "ieeg", "*space-MNI152*_electrodes.tsv"))
    if f:
        for r_ in read_tsv(f[0]):
            try:
                mni[r_["name"]] = [float(r_["x"]), float(r_["y"]), float(r_["z"])]
            except (KeyError, ValueError):
                pass
    pos = {}
    f = glob.glob(os.path.join(root, "derivatives", "epochs", sid, "eeg", "*_electrodes.tsv"))
    if f:
        for r_ in read_tsv(f[0]):
            try:
                pos[r_["name"]] = [float(r_["x"]), float(r_["y"]), float(r_["z"])]
            except (KeyError, ValueError):
                pass
    runs = list(dict.fromkeys(ses))
    d = {"subject": sid, "dataset": "Parmigiani2022", "fs": FS, "level": "primary", "eeg_names": eeg_names,
         "bip_names": bip_names, "eeg_pos": [pos.get(n, [np.nan] * 3) for n in eeg_names],
         "bip_mni": torch.tensor([mni.get(n, [np.nan] * 3) for n in bip_names], dtype=torch.float32),
         "epoch_t0_sample": 60, "norm_clip_z": float("inf"), "norm_stats": {}}
    bounds, c = [], 0
    for e in E:
        bounds.append((c, c + e.shape[1]))
        c += e.shape[1]
    d["trial_bounds"] = torch.tensor(bounds, dtype=torch.long)
    d["trial_session"] = list(ses)
    n_inf = 0
    for k, X in (("eeg_car", np.concatenate(E, 1)), ("ieeg_bip", np.concatenate(Y, 1))):
        med, mad = robust(X)
        z = (X - med) / mad
        n_inf += int((np.abs(z) > 6e4).sum())
        d[k] = torch.from_numpy(np.clip(z, -6e4, 6e4).astype(np.float16))
        st = {"med": med[:, 0].tolist(), "mad": mad[:, 0].tolist()}
        d["norm_stats"][k] = {r_: st for r_ in runs}             # one copy per run (= session in s2i.readout)
    os.makedirs(a.out, exist_ok=True)
    fp = os.path.join(a.out, f"{sid}.pt")
    torch.save(d, fp + ".tmp")
    os.replace(fp + ".tmp", fp)
    torch.save({k: d[k] for k in ("subject", "norm_stats", "norm_clip_z", "trial_bounds", "trial_session")},
               os.path.join(a.out, f"{sid}_normstats.pt"))
    return n_inf


def build(a):
    root = os.path.join(a.src, "ccepcoreg")
    bad = json.load(open(os.path.join(a.src, "eeg_badtrials.json")))
    os.makedirs(a.out, exist_ok=True)
    jp = os.path.join(a.out, "p27_build.json")
    res = json.load(open(jp)) if os.path.exists(jp) else {"patients": {}}
    res["rules"] = {"fs_in": FS_IN, "fs": FS, "cal_frac": CAL_FRAC, "min_epochs": MIN_EPOCHS, "min_eeg": MIN_EEG,
                    "min_targets": MIN_TGT}
    subs = sorted(os.path.basename(p) for p in glob.glob(os.path.join(root, "derivatives", "epochs", "sub-*")))
    for sid in subs:
        if a.patients and sid not in a.patients:
            continue
        if sid in res["patients"] and not a.overwrite:
            log(f"[{sid}] already built")
            continue
        t0 = time.time()
        info, data = build_patient(root, bad, sid)
        if info["excluded"] is None:
            info["float16_overflow_samples"] = write_patient(a, root, sid, data)
        res["patients"][sid] = info
        json.dump(res, open(jp + ".tmp", "w"), indent=1, default=str)
        os.replace(jp + ".tmp", jp)
        log(f"[{sid}] runs {len(info['runs_used'])} (skipped {len(info['runs_skipped'])}), {info.get('n_epochs', 0)} "
            f"epochs (calib {info.get('n_cal', 0)}, test {info.get('n_test', 0)}), EEG {info.get('n_eeg', 0)}, SEEG "
            f"{info.get('n_valid_targets', 0)}, excluded: {info['excluded']} | {time.time() - t0:.0f}s")
    write_build_md(a, res)


def write_build_md(a, res):
    P = res["patients"]
    L = ["# Phase 27 execution log: data (cohort-3 protocol v1.0, Sections 3-5)", "", f"- rules: {res['rules']}", "",
         "| patient | runs used | runs skipped | epochs | calib | test | calib min | test min | EEG ch (listed) | "
         "SEEG targets (listed, near stim) | calib runs | excluded |", "|" + "---|" * 12]
    for sid, i in sorted(P.items()):
        L.append(f"| {sid} | {len(i['runs_used'])} | {len(i['runs_skipped'])} | {i.get('n_epochs', 0)} | "
                 f"{i.get('n_cal', '')} | {i.get('n_test', '')} | {i.get('calib_minutes', 0):.2f} | "
                 f"{i.get('test_minutes', 0):.1f} | {i.get('n_eeg', '')} ({i.get('n_eeg_listed', '')}) | "
                 f"{i.get('n_valid_targets', '')} ({i.get('n_seeg_listed', '')}, {i.get('n_seeg_near_stim', '')}) | "
                 f"{', '.join(i.get('calib_runs', []))} | {i['excluded'] or ''} |")
    inc = [s for s, i in P.items() if i["excluded"] is None]
    L += ["", f"included: {len(inc)} patients: {', '.join(sorted(inc))}", "", "Skipped runs:", ""]
    L += [f"- {sid}: " + "; ".join(f"{r} ({w})" for r, w in i["runs_skipped"]) for sid, i in sorted(P.items())
          if i["runs_skipped"]]
    out = "\n".join(L)
    open(os.path.join(a.out, "p27_build.md"), "w", encoding="utf-8").write(out + "\n")
    print(out)


# ----------------------------------------------------------------------------- runs
def runs(a):
    subs = sorted(os.path.basename(p)[:-3] for p in glob.glob(os.path.join(a.data, "sub-*.pt"))
                  if not p.endswith("_normstats.pt"))
    src = os.path.join(a.enc_root, "short", "enc_final.pt")
    assert os.path.exists(src), f"missing final short encoder {src}"
    od = os.path.join(a.runs, "short")
    os.makedirs(od, exist_ok=True)
    for s in subs:
        dst = os.path.join(od, f"enc_{s}.pt")
        if not os.path.exists(dst):
            shutil.copy(src, dst)
    log(f"short encoder copied for {len(subs)} patients -> {od}")


# ----------------------------------------------------------------------------- report
def _load(run, name):
    out = {}
    for f in glob.glob(os.path.join(run, name, "readout_sub-*.json")):
        d = json.load(open(f))
        out[d["subject"]] = d
    return out


def _ex(D, m, k="r"):
    return {s: r["explicit"][m]["mean"][k] for s, r in D.items() if m in r.get("explicit", {})
            and k in r["explicit"][m]["mean"]}


def compare(a_, b_, rng):
    from scipy.stats import wilcoxon
    s = sorted(set(a_) & set(b_))
    x, y = np.array([a_[q] for q in s]), np.array([b_[q] for q in s])
    d = x - y
    if len(d) == 0:
        return None
    bt = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(10000)])
    try:
        p = float(wilcoxon(x, y, alternative="two-sided").pvalue)
    except ValueError:
        p = float("nan")
    return {"n": len(d), "mean_a": float(x.mean()), "mean_b": float(y.mean()), "median_diff": float(np.median(d)),
            "mean_diff": float(d.mean()), "ci": [float(np.percentile(bt, 2.5)), float(np.percentile(bt, 97.5))],
            "wins": int((d > 0).sum()), "p": p, "per_patient": dict(zip(s, d.round(5).tolist()))}


def holm(ps):
    order = np.argsort(ps)
    adj, prev = [0.0] * len(ps), 0.0
    for rank, j in enumerate(order):
        prev = max(prev, min(1.0, (len(ps) - rank) * ps[j]))
        adj[j] = prev
    return adj


def report(a):
    rng = np.random.default_rng(0)
    R = os.path.join(a.runs, "short")
    B = json.load(open(os.path.join(a.data, "p27_build.json")))["patients"]
    learned, fixed = _load(R, "learned_primary"), _load(R, "fixed_primary")
    included = sorted(s for s, i in B.items() if i["excluded"] is None)
    L = ["# Phase 27: third cohort (Parmigiani et al. 2022), cohort-3 protocol v1.0", "",
         f"- included patients: {len(included)}: {', '.join(included)}"]
    for lab, D in (("learned", learned), ("fixed", fixed)):
        if sorted(D) != included:
            L.append(f"- **WARNING**: {lab} results exist for {len(D)} of {len(included)} patients: run incomplete")
    H = [("H1 (system, r)", "Final-short", "Base", compare(_ex(learned, "combo_nosess"), _ex(learned, "base"), rng)),
         ("H2 (learning, r)", "Learned-short", "Fixed-short",
          compare(_ex(learned, "combo_nosess"), _ex(fixed, "combo_nosess"), rng)),
         ("H3 (system, r_induced_ses)", "Final-short", "Base",
          compare(_ex(learned, "combo_nosess", "r_induced_ses"), _ex(learned, "base", "r_induced_ses"), rng))]
    adj = holm([h[3]["p"] for h in H])
    out = {}
    L += ["", "## Pre-specified hypotheses (two-sided Wilcoxon, Holm over H1-H3)", "",
          "| hypothesis | A | B | n | mean A | mean B | median A − B | mean A − B [95 % CI] | A > B | p | p (Holm) | supported |",
          "|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for (name, la, lb, h), pa in zip(H, adj):
        sup = pa < 0.05 and h["median_diff"] > 0
        h["p_holm"], h["supported"] = pa, bool(sup)
        out[name] = h
        L.append(f"| {name} | {la} | {lb} | {h['n']} | {h['mean_a']:.4f} | {h['mean_b']:.4f} | {h['median_diff']:+.4f} | "
                 f"{h['mean_diff']:+.4f} [{h['ci'][0]:+.4f}, {h['ci'][1]:+.4f}] | {h['wins']}/{h['n']} | {h['p']:.2e} | "
                 f"{pa:.2e} | {'**yes**' if sup else 'no'} |")
    L += ["", "Per-patient differences (A − B):", ""]
    for name, _, _, h in H:
        L.append(f"- {name}: " + ", ".join(f"{s} {v:+.4f}" for s, v in h["per_patient"].items()))
    L += ["", "## Descriptive analyses (unadjusted p)", "",
          "| comparison | metric | n | mean A | mean B | mean A − B [95 % CI] | median A − B | A > B | p |",
          "|---|---|---|---|---|---|---|---|---|"]
    desc = []

    def row(lab, A, B, metric):
        h = compare(A, B, rng)
        if h is None:
            return
        desc.append({"comparison": lab, "metric": metric, **h})
        L.append(f"| {lab} | {metric} | {h['n']} | {h['mean_a']:.4f} | {h['mean_b']:.4f} | {h['mean_diff']:+.4f} "
                 f"[{h['ci'][0]:+.4f}, {h['ci'][1]:+.4f}] | {h['median_diff']:+.4f} | {h['wins']}/{h['n']} | {h['p']:.2e} |")
    for metric in ("r_induced", "r_no50", "r2"):
        row("Final-short − Base", _ex(learned, "combo_nosess", metric), _ex(learned, "base", metric), metric)
    row("Learned-short − Fixed-short", _ex(learned, "combo_nosess", "r_induced_ses"),
        _ex(fixed, "combo_nosess", "r_induced_ses"), "r_induced_ses")
    row("Fixed-short − Base", _ex(fixed, "combo_nosess"), _ex(fixed, "base"), "r")
    row("Learned encoder alone − Base", _ex(learned, "enc"), _ex(learned, "base"), "r")
    l20 = _load(R, "learned_primary_f20")
    if l20:
        row("Final-short, 20 % calibration − Base, 20 %", _ex(l20, "combo_nosess"), _ex(l20, "base"), "r")
        row("Final-short, 20 % − Base, 30 %", _ex(l20, "combo_nosess"), _ex(learned, "base"), "r")
    out["descriptive"] = desc
    L += ["", "Mean over patients:", "", "| model | r | r_induced_ses | r_induced | r_no50 | R² | constant predictions |",
          "|---|---|---|---|---|---|---|"]
    for lab, D, m in (("Base", learned, "base"), ("Fixed-short (+ LAG9)", fixed, "combo_nosess"),
                      ("Learned encoder alone", learned, "enc"), ("Final-short (learned + LAG9)", learned, "combo_nosess")):
        v = [_ex(D, m, k) for k in ("r", "r_induced_ses", "r_induced", "r_no50", "r2")]
        nc = sum(r["explicit"][m].get("n_const", 0) for r in D.values() if m in r.get("explicit", {}))
        L.append(f"| {lab} | " + " | ".join(f"{np.mean(list(x.values())):.4f}" if x else "" for x in v) + f" | {nc} |")
    L += ["", "Per patient (r):", "", "| patient | epochs | calib min | EEG | targets | Base | Fixed-short | Final-short |",
          "|---|---|---|---|---|---|---|---|"]
    fb, ff, fl = _ex(learned, "base"), _ex(fixed, "combo_nosess"), _ex(learned, "combo_nosess")
    for s in included:
        i = B[s]
        L.append(f"| {s} | {i['n_epochs']} | {i['calib_minutes']:.2f} | {i['n_eeg']} | {i['n_valid_targets']} | "
                 f"{fb.get(s, float('nan')):.3f} | {ff.get(s, float('nan')):.3f} | {fl.get(s, float('nan')):.3f} |")
    md = "\n".join(L)
    print(md)
    open(os.path.join(a.runs, "p27_report.md"), "w", encoding="utf-8").write(md + "\n")
    json.dump(out, open(os.path.join(a.runs, "p27_report.json"), "w"), indent=1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=("build", "runs", "report"))
    ap.add_argument("--src", help="build: Phase-26 folder with ccepcoreg/ and eeg_badtrials.json")
    ap.add_argument("--out", help="build: output data folder")
    ap.add_argument("--data", help="runs / report: data folder")
    ap.add_argument("--enc_root", help="runs: folder with short/enc_final.pt (Phase 17 encoders)")
    ap.add_argument("--runs", help="runs / report: run root")
    ap.add_argument("--patients", nargs="*", default=None)
    ap.add_argument("--overwrite", action="store_true")
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    {"build": build, "runs": runs, "report": report}[a.cmd](a)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {CODE_DIR}/s2i/cast37.py
"""
Phase 37 (review 6, C2; post hoc) - the CAST architecture of our earlier preprint (Pham and Tran 2026, arXiv
2605.18897) re-run under the protocol of the present study, on ds004752.

Kept from CAST (its code, model/cast.py and the ds004752 LOSO script): the architecture (multi-scale patches 8/16/32,
d = 128, 4 heads, 4 encoder and 2 decoder layers, dropout 0.2), the loss (MSE + (1 - r) + 0.1 x log-spectral), 2-s
windows (400 samples) with a 1-s stride for training, per-window z-scoring of the windows used for training, the
augmentation, leave-one-patient-out sequential encoder pre-training (2 rounds x 50 epochs per training patient,
AdamW lr 3e-4, weight decay 0.01, batch 16), and the decoder calibration on all contacts (200 epochs, encoder fine-tuned
at a tenth of the learning rate, best epoch by training loss).
Replaced by the protocol of the present study: bipolar targets and CAR scalp EEG (primary level, all 15 patients, all
contacts), strict normalisation (calibration statistics only, data of s2i.strict36), chronological calibration on the
first max(3, round(0.2 n)) trials with a one-trial gap, and the explicit-prediction r of every contact over the test
trials (s2i.metrics.per_contact_metrics). Test trials are tiled with 2-s windows (the last one aligned with the trial
end), each window is z-scored on its scalp EEG only, and overlapping predictions are averaged.
Also reported on the same predictions: CAST's own metric (each 2-s test window and its target z-scored separately, r on
the concatenated windows).

python -m s2i.cast37 run    --data DATA36 --out RUNS/p37_cast
python -m s2i.cast37 report --runs RUNS
"""
import argparse
import glob
import json
import math
import os
import random
import sys
import time

import numpy as np

W, STRIDE, MAX_EEG = 400, 200, 21
ENC_KEYS = ("patch_embeds.", "scale_embed", "pos_encs.", "encoder.")


def build_model_class():
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    class SinCos(nn.Module):
        def __init__(self, d, max_len=256):
            super().__init__()
            pe = torch.zeros(max_len, d)
            pos = torch.arange(max_len).unsqueeze(1).float()
            div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
            pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
            self.register_buffer("pe", pe.unsqueeze(0))

        def forward(self, x):
            return x + self.pe[:, :x.size(1)]

    def pearson_loss(p, t):
        vx, vy = p - p.mean(-1, keepdim=True), t - t.mean(-1, keepdim=True)
        return (1 - (vx * vy).sum(-1) / torch.sqrt((vx ** 2).sum(-1) * (vy ** 2).sum(-1) + 1e-8)).mean()

    def spectral_loss(p, t):
        return F.mse_loss(torch.log(torch.fft.rfft(p, dim=-1).abs() + 1e-8),
                          torch.log(torch.fft.rfft(t, dim=-1).abs() + 1e-8))

    class CAST(nn.Module):
        """Copy of model/cast.py (CAST v3) of the earlier preprint."""

        def __init__(self, c_eeg=21, c_ieeg=64, seq_len=400, d_model=128, n_heads=4, n_enc_layers=4,
                     n_dec_layers=2, dropout=0.2):
            super().__init__()
            self.c_ieeg, self.seq_len, self.d_model = c_ieeg, seq_len, d_model
            self.patch_sizes = [8, 16, 32]
            self.n_patches_per_scale = [seq_len // ps for ps in self.patch_sizes]
            self.total_tokens = sum(self.n_patches_per_scale)
            self.patch_embeds = nn.ModuleList([nn.Sequential(
                nn.Linear(c_eeg * ps, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(d_model, d_model), nn.LayerNorm(d_model)) for ps in self.patch_sizes])
            self.scale_embed = nn.Parameter(torch.randn(len(self.patch_sizes), 1, d_model) * 0.02)
            self.pos_encs = nn.ModuleList([SinCos(d_model, n + 1) for n in self.n_patches_per_scale])
            self.encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(
                d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4, dropout=dropout, activation="gelu",
                batch_first=True, norm_first=True), num_layers=n_enc_layers)
            n_q = self.n_patches_per_scale[1]
            self.temporal_queries = nn.Parameter(torch.randn(1, n_q, d_model) * 0.02)
            self.channel_embed = nn.Embedding(c_ieeg, d_model)
            self.query_pos = SinCos(d_model, n_q + 1)
            self.decoder = nn.TransformerDecoder(nn.TransformerDecoderLayer(
                d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4, dropout=dropout, activation="gelu",
                batch_first=True, norm_first=True), num_layers=n_dec_layers)
            self.unpatch = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout),
                                         nn.Linear(d_model, 16))

        def forward(self, eeg):
            B = eeg.shape[0]
            toks = []
            for i, ps in enumerate(self.patch_sizes):
                p = eeg.unfold(2, ps, ps).permute(0, 2, 1, 3).reshape(B, self.n_patches_per_scale[i], -1)
                toks.append(self.pos_encs[i](self.patch_embeds[i](p)) + self.scale_embed[i])
            memory = self.encoder(torch.cat(toks, 1))
            n_q = self.temporal_queries.shape[1]
            mem = memory.unsqueeze(1).expand(B, self.c_ieeg, -1, -1).reshape(B * self.c_ieeg, self.total_tokens, -1)
            q = self.query_pos(self.temporal_queries + self.channel_embed(
                torch.arange(self.c_ieeg, device=eeg.device)).unsqueeze(1))
            q = q.unsqueeze(0).expand(B, -1, -1, -1).reshape(B * self.c_ieeg, n_q, self.d_model)
            out = self.unpatch(self.decoder(q, mem)).reshape(B, self.c_ieeg, -1)
            return out[:, :, :self.seq_len]

        def compute_loss(self, eeg, ieeg):
            pred = self.forward(eeg)
            return F.mse_loss(pred, ieeg) + pearson_loss(pred, ieeg) + 0.1 * spectral_loss(pred, ieeg)

    return CAST


def zs(x):
    return (x - x.mean(-1, keepdims=True)) / (x.std(-1, keepdims=True) + 1e-6)


def pad_eeg(e):
    return np.concatenate([e, np.zeros((MAX_EEG - e.shape[0], e.shape[1]), e.dtype)]) if e.shape[0] < MAX_EEG else e


def train_windows(E, Y):
    xs, ys = [], []
    for e, y in zip(E, Y):
        for s in range(0, e.shape[1] - W + 1, STRIDE):
            xs.append(zs(pad_eeg(e[:, s:s + W])))
            ys.append(zs(y[:, s:s + W]))
    return np.stack(xs).astype(np.float32), np.stack(ys).astype(np.float32)


def starts(L):
    s = list(range(0, L - W + 1, W))
    if s and s[-1] + W < L:
        s.append(L - W)
    return s


def windows_index(E):
    """(trial, start) of every training-style window (2 s, 1-s stride) of a patient."""
    return [(k, s) for k, e in enumerate(E) for s in range(0, e.shape[1] - W + 1, STRIDE)]


def load(path, eeg_key="eeg_car", tgt_key="ieeg_bip"):
    import torch
    d = torch.load(path, weights_only=False)
    eeg, tgt = d[eeg_key].float().numpy(), d[tgt_key].float().numpy()
    b = d["trial_bounds"].numpy()
    return d["subject"], [eeg[:, s0:s1] for s0, s1 in b], [tgt[:, s0:s1] for s0, s1 in b]


def run(a):
    import torch
    import torch.nn as nn
    from torch.optim import AdamW
    from torch.optim.lr_scheduler import CosineAnnealingLR
    from torch.utils.data import DataLoader, TensorDataset
    from s2i.metrics import per_contact_metrics
    CAST = build_model_class()
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(a.out, exist_ok=True)
    seed = 42

    def fix(s):
        random.seed(s)
        np.random.seed(s)
        torch.manual_seed(s)
        torch.cuda.manual_seed_all(s)

    def augment(e, i):
        e, i = e + torch.randn_like(e) * 0.05, i + torch.randn_like(i) * 0.05
        s = 0.8 + 0.4 * torch.rand(e.shape[0], 1, 1, device=e.device)
        sh = torch.randint(-5, 6, (1,)).item()
        if sh:
            e, i = torch.roll(e, sh, -1), torch.roll(i, sh, -1)
        return e * s, i * s

    def model(n_ieeg):
        return CAST(c_eeg=MAX_EEG, c_ieeg=n_ieeg, seq_len=W, d_model=128, n_heads=4, n_enc_layers=4,
                    n_dec_layers=2, dropout=0.2).to(dev)

    def enc_of(m):
        return {k: v.detach().cpu().clone() for k, v in m.state_dict().items() if k.startswith(ENC_KEYS)}

    def loader(x, y, s):
        bs = min(16, len(x))
        return DataLoader(TensorDataset(torch.from_numpy(x), torch.from_numpy(y)), batch_size=bs, shuffle=True,
                          drop_last=len(x) > bs, generator=torch.Generator().manual_seed(s))

    subs = {}
    for p in sorted(glob.glob(os.path.join(a.data, "sub-[0-9][0-9].pt"))):
        sid, E, Y = load(p, "eeg_car" if a.eeg == "car" else "eeg_orig", "ieeg_bip" if a.target == "bip" else "ieeg_mono")
        subs[sid] = (E, Y)
        print(f"{sid}: {len(E)} trials, {E[0].shape[0]} scalp, {Y[0].shape[0]} targets", flush=True)
    order = sorted(subs)
    for tsid in (a.only or order):                     # --only: folds run in the given order
        ti = order.index(tsid)                         # seeds depend on the patient, not on the run order
        fp = os.path.join(a.out, f"cast_{tsid}.json")
        if os.path.exists(fp):
            print(f"[{tsid}] done", flush=True)
            continue
        t0 = time.time()
        E, Y = subs[tsid]
        n = len(E)
        n_cal = max(3, int(round(n * 0.2)))
        cal, test = list(range(n_cal)), list(range(n_cal + 1, n))
        # ---- encoder: sequential pre-training on the other patients (resumable after every patient)
        ep = os.path.join(a.out, f"enc_{tsid}.pt")
        prog = os.path.join(a.out, f"progress_{tsid}.pt")
        train_sids = [s for s in order if s != tsid]
        if os.path.exists(ep):
            enc_state = torch.load(ep, weights_only=False)
        else:
            fix(seed + ti)
            base = model(Y[0].shape[0])
            done = 0
            if os.path.exists(prog):
                pr = torch.load(prog, weights_only=False)
                base.load_state_dict(pr["enc"], strict=False)
                done = pr["done"]
            steps = [(r_, s) for r_ in range(a.rounds) for s in train_sids]
            for k, (round_i, sid) in enumerate(steps):
                if k < done:
                    continue
                Es, Ys = subs[sid]
                x, y = train_windows(Es, Ys)
                tmp = model(Ys[0].shape[0])
                tmp.load_state_dict(enc_of(base), strict=False)
                opt = AdamW(tmp.parameters(), lr=3e-4, weight_decay=0.01)
                dl = loader(x, y, seed + round_i)
                for _ in range(a.epochs_pre):
                    tmp.train()
                    for e, i in dl:
                        e, i = augment(e.to(dev), i.to(dev))
                        loss = tmp.compute_loss(e, i)
                        opt.zero_grad()
                        loss.backward()
                        nn.utils.clip_grad_norm_(tmp.parameters(), 1.0)
                        opt.step()
                base.load_state_dict(enc_of(tmp), strict=False)
                del tmp, opt
                torch.cuda.empty_cache()
                torch.save({"enc": enc_of(base), "done": k + 1}, prog + ".tmp")
                os.replace(prog + ".tmp", prog)
                print(f"  [{tsid}] pre-training {k + 1}/{len(steps)} ({sid}, round {round_i + 1}) | "
                      f"{(time.time() - t0) / 60:.1f} min", flush=True)
            enc_state = enc_of(base)
            torch.save(enc_state, ep)
            if os.path.exists(prog):
                os.remove(prog)
            del base
        # ---- decoder calibration on all contacts (CAST "ALL channels" setting)
        fix(seed + ti * 100)
        K = Y[0].shape[0]
        m = model(K)
        m.load_state_dict(enc_state, strict=False)
        enc_p = [p for nme, p in m.named_parameters() if nme.startswith(ENC_KEYS)]
        dec_p = [p for nme, p in m.named_parameters() if not nme.startswith(ENC_KEYS)]
        opt = AdamW([{"params": enc_p, "lr": 3e-5}, {"params": dec_p, "lr": 3e-4}], weight_decay=0.01)
        sched = CosineAnnealingLR(opt, T_max=a.epochs_dec)
        if a.split == "random_windows":
            # CAST's own split: every 2-s window (1-s stride) of the patient, permuted with seed 42, first 20 % for
            # calibration (int(0.2 N), at least 3), the rest for testing
            idx = windows_index(E)
            xa, ya = train_windows(E, Y)
            perm = np.random.RandomState(42).permutation(len(idx))
            n_cw = max(3, int(len(idx) * 0.2))
            cw, tw = perm[:n_cw], perm[n_cw:]
            x, y = xa[cw], ya[cw]
            calset = {idx[j] for j in cw}
            shared = float(np.mean([any((idx[j][0], idx[j][1] + d) in calset for d in (-STRIDE, STRIDE)) for j in tw]))
        else:
            x, y = train_windows([E[i] for i in cal], [Y[i] for i in cal])
        dl = loader(x, y, seed)
        best, best_st = float("inf"), None
        for _ in range(a.epochs_dec):
            m.train()
            tot = 0.0
            for e, i in dl:
                e, i = augment(e.to(dev), i.to(dev))
                loss = m.compute_loss(e, i)
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step()
                tot += loss.item()
            sched.step()
            if tot / len(dl) < best:
                best, best_st = tot / len(dl), {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
        m.load_state_dict(best_st)
        m.eval()
        if a.split == "random_windows":                     # CAST's metric: r of the concatenated z-scored windows
            with torch.no_grad():
                xt = torch.from_numpy(xa[tw])
                pw = np.concatenate([m(xt[j:j + 8].to(dev)).cpu().numpy() for j in range(0, len(tw), 8)])
            r_cast = per_contact_metrics([np.concatenate(list(pw), 1)], [np.concatenate(list(ya[tw]), 1)], 200,
                                         bands=False)["r"]
            res = {"subject": tsid, "split": "random_windows", "target": a.target, "eeg": a.eeg,
                   "n_windows": len(idx), "n_cal_windows": int(n_cw), "test_windows_sharing_1s_with_calibration": shared,
                   "n_targets": K, "r_cast_metric": float(np.nanmean(r_cast)), "minutes": (time.time() - t0) / 60}
            json.dump(res, open(fp + ".tmp", "w"))
            os.replace(fp + ".tmp", fp)
            print(f"[{tsid}] CAST protocol: r = {res['r_cast_metric']:.4f} | test windows sharing 1 s with "
                  f"calibration {shared:.2f} | {res['minutes']:.1f} min", flush=True)
            del m, opt
            torch.cuda.empty_cache()
            continue
        # ---- prediction of the test trials (2-s tiles, scalp EEG z-scored per window, overlaps averaged)
        P, Tt, wp, wt = [], [], [], []
        with torch.no_grad():
            for i in test:
                e, t = E[i], Y[i]
                L = e.shape[1]
                ss = starts(L)
                if not ss:
                    continue
                xw = torch.from_numpy(np.stack([zs(pad_eeg(e[:, s:s + W])) for s in ss]).astype(np.float32)).to(dev)
                pw = np.concatenate([m(xw[j:j + 8]).cpu().numpy() for j in range(0, len(ss), 8)])
                acc, cnt = np.zeros((K, L)), np.zeros(L)
                for s, p in zip(ss, pw):
                    acc[:, s:s + W] += p
                    cnt[s:s + W] += 1
                cov = cnt > 0
                P.append((acc[:, cov] / cnt[cov]).astype(np.float32))
                Tt.append(t[:, cov])
                for s, p in zip(ss, pw):
                    if s % W == 0:                                               # non-overlapping tiles only
                        wp.append(p)
                        wt.append(zs(t[:, s:s + W]))
        r_ours = per_contact_metrics(P, Tt, 200, bands=False)["r"]
        Wp, Wt = np.concatenate(wp, 1), np.concatenate(wt, 1)
        r_cast = per_contact_metrics([Wp], [Wt], 200, bands=False)["r"]
        res = {"subject": tsid, "n_trials": n, "n_cal": n_cal, "n_test": len(test), "n_targets": K,
               "r": float(np.nanmean(r_ours)), "r_cast_metric": float(np.nanmean(r_cast)),
               "r_contacts": [float(v) for v in r_ours], "minutes": (time.time() - t0) / 60}
        json.dump(res, open(fp + ".tmp", "w"))
        os.replace(fp + ".tmp", fp)
        print(f"[{tsid}] r = {res['r']:.4f} (CAST metric {res['r_cast_metric']:.4f}) | {res['minutes']:.1f} min",
              flush=True)
        del m, opt
        torch.cuda.empty_cache()


def report(a):
    from s2i.parm27 import _ex, _load, compare
    rng = np.random.default_rng(0)
    R = a.runs
    cast = {json.load(open(f))["subject"]: json.load(open(f)) for f in glob.glob(os.path.join(R, "p37_cast", "cast_sub-*.json"))}
    x = {s: v["r"] for s, v in cast.items()}
    xm = {s: v["r_cast_metric"] for s, v in cast.items()}
    fin = _load(os.path.join(R, "shared_bip_car"), "readout_combo5_strict_qc_none")
    base, final = _ex(fin, "base"), _ex(fin, "combo_nosess")
    L = ["# Phase 37: CAST re-run under the present protocol (ds004752, primary, strict; post hoc, unadjusted p)", "",
         f"- patients: {len(x)}; CAST mean r {np.mean(list(x.values())):.4f}; with its own metric on the same "
         f"predictions {np.mean(list(xm.values())):.4f}", "",
         "| A | B | n | mean A | mean B | A − B [95 % CI] | A > B | p |", "|---|---|---|---|---|---|---|---|"]
    out = {"mean_r": float(np.mean(list(x.values()))), "mean_r_cast_metric": float(np.mean(list(xm.values())))}
    for lab, A, B in (("CAST − LAG9 ridge", x, base), ("final model − CAST", final, x),
                      ("CAST metric − present metric (same predictions)", xm, x)):
        h = compare(A, B, rng)
        out[lab] = h
        L.append(f"| {lab.split(' − ')[0]} | {lab.split(' − ')[1]} | {h['n']} | {h['mean_a']:.4f} | {h['mean_b']:.4f} | "
                 f"{h['mean_diff']:+.4f} [{h['ci'][0]:+.4f}, {h['ci'][1]:+.4f}] | {h['wins']}/{h['n']} | {h['p']:.2e} |")
    L += ["", "Per patient (r): " + ", ".join(f"{s} {v:.3f}" for s, v in sorted(x.items()))]
    ctrl = [json.load(open(f)) for f in sorted(glob.glob(os.path.join(R, "p37_cast_origprotocol", "cast_sub-*.json")))]
    reported = {"sub-01": 0.268, "sub-13": 0.784}      # CAST preprint, ds004752 LOSO, all channels
    L += ["", "Positive control (same code, CAST's own protocol: monopolar, original reference, random split of "
              "overlapping 2-s windows, CAST metric):"]
    for c in ctrl:
        L.append(f"- {c['subject']}: r = {c['r_cast_metric']:.3f} (CAST preprint {reported.get(c['subject'], float('nan')):.3f}); "
                 f"test windows sharing 1 s with calibration {c['test_windows_sharing_1s_with_calibration']:.2f}")
        out[f"control {c['subject']}"] = c
    if not ctrl:
        L.append("- not run yet")
    md = "\n".join(L)
    print(md)
    open(os.path.join(R, "cast37_report.md"), "w", encoding="utf-8").write(md + "\n")
    json.dump(out, open(os.path.join(R, "cast37_report.json"), "w"), indent=1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=("run", "report"))
    ap.add_argument("--data")
    ap.add_argument("--out")
    ap.add_argument("--runs")
    ap.add_argument("--rounds", type=int, default=2)
    ap.add_argument("--epochs_pre", type=int, default=50)
    ap.add_argument("--epochs_dec", type=int, default=200)
    ap.add_argument("--only", nargs="*", default=None)
    ap.add_argument("--target", choices=("bip", "mono"), default="bip")
    ap.add_argument("--eeg", choices=("car", "orig"), default="car")
    ap.add_argument("--split", choices=("chrono", "random_windows"), default="chrono",
                    help="random_windows (with --target mono --eeg orig): positive control with CAST's own protocol")
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    run(a) if a.cmd == "run" else report(a)


if __name__ == "__main__":
    main()


In [ ]:
#@title 37A. Chạy (một tiến trình; tự bỏ qua phần đã xong; chạy lại cell này nếu bị ngắt)
# 0) dừng mọi tiến trình CAST còn sót từ lần chạy song song trước
!pkill -f "s2i.cast37" ; sleep 3 ; echo "đã dọn các tiến trình cũ"
# 1) ĐỐI CHỨNG DƯƠNG (chạy trước): protocol gốc của CAST trên sub-01 và sub-13
!python -W ignore -m s2i.cast37 run --data {NOQC} --out {RUNS_DIR}/p37_cast_origprotocol --only sub-01 sub-13 --target mono --eeg orig --split random_windows 2>&1 | grep -E "^\[|Error|error"
# 2) CAST theo protocol của bài, các fold còn lại theo thứ tự ưu tiên (đã xong: 01, 02, 03, 04, 05, 06)
!python -W ignore -m s2i.cast37 run --data {DATA36} --out {RUNS_DIR}/p37_cast --only sub-07 sub-08 sub-09 sub-13 sub-15 sub-10 sub-11 sub-12 sub-14 2>&1 | grep -E "^\[|Error|error"

In [ ]:
#@title 37B. Report (CAST theo protocol mới + đối chứng dương)
!python -m s2i.cast37 report --runs {RUNS_DIR}

Gửi tôi output của **37B** sau khi đối chứng dương xong (có 2 dòng `Positive control`).